# DATA COLLECTION NOTEBOOK 

### Objectives

* Fetch data from Kaggle dataset . The datasource for this project has been provided by cCode institute at https://www.kaggle.com/datasets/codeinstitute/cherry-leaves in the form a zip file 
* Extract the Zip file , and prepare it for further Machine Learning analysis 
* Save the file in out/dataset/ folder and push it to the repo 
  

### Inputs

* The input for this notebook is a Kaggle dataset from Code Insittute at https://www.kaggle.com/datasets/codeinstitute/cherry-leaves
* This zip file is saved and extracted at input/datase/cherry_leaves/folder 
* There are Two  file - Healthy and power_mildew

### Outputs

* The output will stored in the output/dataset folder and pushed to the gitpod repo.  


### Machine Learning Pipleine
 
  - A typical workflow used for supervised learning is: 
     - Split the dataset into train and test set
     - Fit the model (either using a pipeline or not)
     - Evaluate your model. 
       - If performance is not good,revisit the process, 
         - start from data collection
         - Conduct EDA (Exploratory Data Analysis) etc.
 
  - The machine learning piepline is a sequence of operations that are performed when training a machine learning model
  - In this notebook we complete the following tasks. 
   
    #### Data Collection

     - We collect data supplied by the client from the kaggle website and store it in the input folder.
   
     #### Data Cleaning or Correcting 

    - Because the dataset was provided by the client does not mke it valid and accurate.
    - Remember garbage in => garbage out. 
    - We complete this task in section 3 of this notebook.
    - It involves analysisng the data to remove non image files
    - Remember: This is the most time consuming step in the ML process
  
    #### Feature Engineering 
    - This task overlaps with data cleaning or correcting tasks above.
    - The task of dropping missing data  ( remove non-image files) was completed in section 3. 
    - We did not carry any specific tasks related to feature engineering like imputing, binning, tidy data or date extraction.



 Install Requirements.txt file

In [1]:
%pip install -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.


Import the numpy package

In [2]:
import numpy

---

## Change working directory 

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [3]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Roshan Learning\\Project5\\Project5_Mildew_Detection_in_Cherry_Leaves\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [4]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [5]:
current_dir = os.getcwd()
current_dir

'c:\\Roshan Learning\\Project5\\Project5_Mildew_Detection_in_Cherry_Leaves'

# Section 1- Download the Kaggle Dataset

Section 1.1 Install the Kaggle package 

In [6]:
 ! pip install kaggle

  Using cached bleach-6.2.0-py3-none-any.whl.metadata (30 kB)
  Using cached webencodings-0.5.1-py2.py3-none-any.whl.metadata (2.1 kB)
Using cached bleach-6.2.0-py3-none-any.whl (163 kB)
Using cached webencodings-0.5.1-py2.py3-none-any.whl (11 kB)

   ------------- -------------------------- 2/6 [tqdm]
   -------------------------- ------------- 4/6 [bleach]
   --------------------------------- ------ 5/6 [kaggle]
   --------------------------------- ------ 5/6 [kaggle]
   ---------------------------------------- 6/6 [kaggle]



In [7]:
! pip install --upgrade pip


---

Section 2 Unzip the archive file in the input/dataset directory 

In [11]:

import zipfile
import os

kaggleDatasetpath = "input/dataset/archive.zip"
destinationFolder = "input/dataset"

if os.path.exists(kaggleDatasetpath):
    with zipfile.ZipFile(kaggleDatasetpath, 'r') as zip_ref:
        zip_ref.extractall(destinationFolder)
    print("Archive extracted successfully.")
else:
    print("ZIP file not found:", kaggleDatasetpath)


Archive extracted successfully.


# Section 3 - Data Cleaning 

### Section3.1 Data Preparation 

 Check and Remove non-image files 

In [12]:
def remove_non_image_file(my_data_dir):
    image_extension = ('.png', '.jpg', '.jpeg')
    folders = os.listdir(my_data_dir)
    for folder in folders:
        files = os.listdir(my_data_dir + '/' + folder)
        # print(files)
        i = []
        j = []
        for given_file in files:
            if not given_file.lower().endswith(image_extension):
                file_location = my_data_dir + '/' + folder + '/' + given_file
                os.remove(file_location)  # remove non image file
                i.append(1)
            else:
                j.append(1)
                pass
        print(f"Folder: {folder} - has image file", len(j))
        print(f"Folder: {folder} - has non-image file", len(i))

---

**Split train validation test set**
- in what ratio should we split the data for Train, Test and Validation?
    - As a rule of thumb, with **supervised learning**
      - 20-30% of the data is used for the test set.
      - 10-20% for  validation sets 
      - Balance being set aside for the train set.
    - The Validation and test sets are similar in  that they are both used to evaluate the model performance.
    - However, they differ only in that the  validation set is used in iteratively tuning the  model for improved performance, through comparison  of different algorithms or hyperparameters.
    - Therefore, as it is no longer considered  to be unseen data by the model,  it can’t be used again as part of the test set.  

In [13]:
import os
import shutil
import random
import joblib


def split_train_validation_test_images(my_data_dir, train_set_ratio, validation_set_ratio, test_set_ratio):

    if train_set_ratio + validation_set_ratio + test_set_ratio != 1.0:
        print("train_set_ratio + validation_set_ratio + test_set_ratio should sum to 1.0")
        return

    # gets classes labels
    labels = os.listdir(my_data_dir)  # it should get only the folder name
    if 'test' in labels:
        pass
    else:
        # create train, test folders with classes labels sub-folder
        for folder in ['train', 'validation', 'test']:
            for label in labels:
                os.makedirs(name=my_data_dir + '/' + folder + '/' + label)

        for label in labels:

            files = os.listdir(my_data_dir + '/' + label)
            random.shuffle(files)

            train_set_files_qty = int(len(files) * train_set_ratio)
            validation_set_files_qty = int(len(files) * validation_set_ratio)

            count = 1
            for file_name in files:
                if count <= train_set_files_qty:
                    # move a given file to the train set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/train/' + label + '/' + file_name)

                elif count <= (train_set_files_qty + validation_set_files_qty):
                    # move a given file to the validation set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/validation/' + label + '/' + file_name)

                else:
                    # move given file to test set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/test/' + label + '/' + file_name)

                count += 1

            os.rmdir(my_data_dir + '/' + label)

Conventionally,

- The training set is divided into a 0.70 ratio of data.
- The validation set is divided into a 0.10 ratio of data.
- The test set is divided into a 0.20 ratio of data.

In [15]:
split_train_validation_test_images(my_data_dir=f"input/dataset/cherry-leaves",
                                   train_set_ratio=0.7,
                                   validation_set_ratio=0.1,
                                   test_set_ratio=0.2
                                   )

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Next Steps- ML Pipeline 

The machine learning piepline is a sequence of operations that are performed when training a machine learning model. 

-   In this notebook we completed the following tasks.
    - Data Collection:  
    - Data Cleaning or Correcting
    - Feature Engineering
- In the next note book, DataVisualization we carry out Data Augmentation.
  